In [3]:
import numpy as np
from schemas import GeneratorTech, SystemParameters, ExpansionInput
from CE_model import build_and_solve_expansion

def run_test():
    print("🚀 Προετοιμασία δοκιμαστικών δεδομένων...")

    # 1. Δημιουργία χρονοσειράς ζήτησης (24 ώρες * 14 ημέρες = 336 ώρες για γρήγορο test)
    # Χρησιμοποιούμε μια ημιτονοειδή καμπύλη με αιχμές
    hours = 336
    t = np.arange(hours)
    # Βασική ζήτηση 2.000 MW με ημερήσια διακύμανση ±800 MW
    base_demand = 2000 + 800 * np.sin(2 * np.pi * t / 24) + np.random.normal(0, 50, hours)
    demand_profile = np.maximum(base_demand, 500).tolist()

    # 2. Capacity factors για ΑΠΕ (Solar & Wind)
    # Solar: Διακύμανση κατά τη διάρκεια της ημέρας (μηδέν τη νύχτα)
    solar_cf = np.maximum(0, np.sin(2 * np.pi * (t - 6) / 24)).tolist()
    # Wind: Πιο τυχαία διακύμανση με μέσο όρο ~0.35
    wind_cf = np.clip(0.35 + 0.25 * np.sin(2 * np.pi * t / 100) + np.random.normal(0, 0.1, hours), 0, 1).tolist()

    # 3. Ορισμός Τεχνολογιών
    technologies = [
        GeneratorTech(
            name="CCGT_Base",
            is_variable=False,
            capex_per_mw=80000.0,      # High CAPEX
            om_fixed_per_mw=15000.0,
            var_cost_per_mwh=45.0,      # Low Var Cost (High efficiency gas)
            co2_tons_per_mwh=0.35,
            existing_capacity_mw=1000.0,
            max_new_capacity_mw=2000.0
        ),
        GeneratorTech(
            name="OCGT_Peaker",
            is_variable=False,
            capex_per_mw=45000.0,      # Low CAPEX
            om_fixed_per_mw=8000.0,
            var_cost_per_mwh=95.0,      # High Var Cost (Peaker)
            co2_tons_per_mwh=0.55,
            existing_capacity_mw=300.0,
            max_new_capacity_mw=1500.0
        ),
        GeneratorTech(
            name="Solar_PV",
            is_variable=True,
            capex_per_mw=50000.0,      # Low CAPEX
            om_fixed_per_mw=7000.0,
            var_cost_per_mwh=0.0,       # Zero fuel cost
            co2_tons_per_mwh=0.0,
            existing_capacity_mw=500.0,
            max_new_capacity_mw=3000.0
        ),
        GeneratorTech(
            name="Wind_Onshore",
            is_variable=True,
            capex_per_mw=75000.0,
            om_fixed_per_mw=12000.0,
            var_cost_per_mwh=0.0,
            co2_tons_per_mwh=0.0,
            existing_capacity_mw=400.0,
            max_new_capacity_mw=2000.0
        ),
    ]

    # 4. Παράμετροι Συστήματος
    system_params = SystemParameters(
        prm_margin=0.15,            # 15% Reserve Margin
        co2_cap_tons=150000.0,       # Όριο CO2 για την περίοδο
        voLL=3000.0
    )

    # 5. Σύνθεση Εισόδου (Validation μέσω Pydantic)
    inputs = ExpansionInput(
        system_params=system_params,
        technologies=technologies,
        demand_profile=demand_profile,
        capacity_factors={
            "Solar_PV": solar_cf,
            "Wind_Onshore": wind_cf
        }
    )

    print("⚙️ Εκτέλεση Optimization Engine via Pyomo (Highs / GLPK solver)...")
    
    # Εκτέλεση (χρησιμοποιεί CBC, HiGHS ή GLPK ανάλογα τι έχετε εγκατεστημένο)
    # Δοκιμάστε solver_name="appsi_highs", "glpk" ή "cbc"
    try:
        results = build_and_solve_expansion(inputs, solver_name="glpk")
    except Exception as e:
        print(f"❌ Σφάλμα κατά την επίλυση: {e}")
        return

    # 6. Εκτύπωση Αποτελεσμάτων
    print("\n" + "="*50)
    print("📊 ΑΠΟΤΕΛΕΣΜΑΤΑ CAPACITY EXPANSION OPTIMIZATION")
    print("="*50)
    print(f"Status Solver:         {results.status}")
    print(f"Συνολικό Κόστος:       €{results.total_cost:,.2f}")
    print(f"Μη Εξυπηρετούμενη Ενέργεια: {results.unserved_energy_mwh:.2f} MWh")
    print(f"Συνολικές Εκπομπές CO2: {results.co2_emissions_tons:,.2f} Tons")
    print("-" * 50)
    print("🏗️  ΕΓΚΑΤΕΣΤΗΜΕΝΗ ΙΣΧΥΣ (TOTAL CAPACITIES):")
    for tech, cap in results.built_capacities.items():
        print(f"  • {tech:<15}: {cap:,.2f} MW")
    
    print("-" * 50)
    print("DETAILS PER TECHNOLOGY:")
    print(f"{'Tech':<15} | {'Capacity (MW)':<15} | {'Generation (MWh)':<18} | {'Annual Cost (€)':<15}")
    print("-" * 68)
    for d in results.details:
        print(f"{d.tech_name:<15} | {d.capacity_mw:<15,.2f} | {d.total_generation_mwh:<18,.2f} | €{d.annual_cost:,.2f}")
    print("="*50)

if __name__ == "__main__":
    run_test()

ModuleNotFoundError: No module named 'pyomo'